# 📘 Aula 07c: Visão Computacional com PyTorch – MNIST Flat & Pipelines Scikit-Learn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fboldt/aulasann/blob/main/aula07c_mnist_flat_torch.ipynb)

Após explorarmos o MNIST com Keras nas aulas 07a e 07b, daremos início à transição correspondente no ecossistema **PyTorch**.

Neste caderno, implementamos o modelo **MLP Flat (Totalmente Conectado)** utilizando as ferramentas canônicas do PyTorch: classes herdadas de `nn.Module`, conjuntos de dados com `TensorDataset`, gerenciamento de lotes e paralelismo com `DataLoader`, funções de custo (`nn.CrossEntropyLoss`) e laço de treinamento explícito com autodiferenciação. Além disso, criaremos o wrapper profissional `TorchWrappedNN` para integrar o PyTorch diretamente aos Pipelines do Scikit-Learn!

---

### 🎯 Objetivo da Aula
1. Compreender o ecossistema de dados do PyTorch: tensores, tipos primitivos (`.float()` vs `.long()`) e alocação dinâmica em dispositivo (`cpu` ou `cuda`).
2. Dominar as classes `TensorDataset` e `DataLoader` para embaralhamento e carregamento de minibatches em memória.
3. Implementar a classe `BasicTorchNN(nn.Module)` definindo `__init__` e o método de propagação direta `forward`.
4. Compreender por que a camada final no PyTorch não precisa de Softmax explícita quando combinada com `nn.CrossEntropyLoss()`.
5. Escrever o laço de treinamento explícito com minibatches e avaliação vetorial em teste sob `torch.no_grad()`.
6. Construir a classe `TorchWrappedNN` compatível com a API do Scikit-Learn (`fit` e `predict`) e integrá-la a um `Pipeline` com `Divide255` e `Flatten`.

---

### 💡 A Intenção Pedagógica do Professor
* **Controle de Baixo Nível vs Produtividade**: Enquanto o Keras oculta os detalhes do laço de treino, o PyTorch expõe explicitamente cada chamada (`zero_grad`, `forward`, `backward`, `step`). O professor apresenta ambos os paradigmas para que o aluno domine tanto a visão procedural quanto declarativa.
* **Estabilidade Numérica da Perda**: Destacar que `nn.CrossEntropyLoss()` espera *logits* brutos (sem Softmax), pois computa o log-sum-exp internamente para evitar estouro numérico (*overflow/underflow*).
* **Unificação de Frameworks**: Demonstrar que o padrão de design `Pipeline` do Scikit-Learn é universal: o mesmo pipeline aceita Keras, PyTorch ou Random Forest!

---

### 📚 Glossário da Aula
* **`torch.device`**: Identificador de hardware que direciona tensores para processamento em GPU (`cuda:0`) ou CPU.
* **`TensorDataset`**: Empacotador que agrupa tensores de características ($X$) e rótulos ($y$) como um dataset indexável.
* **`DataLoader`**: Utilitário que automatiza o particionamento em minibatches, embaralhamento (`shuffle=True`) e carregamento multi-thread.
* **`nn.Module`**: Classe fundamental do PyTorch da qual todas as redes neurais e camadas derivam.
* **`torch.no_grad()`**: Context manager que desliga o mecanismo autograd durante a validação ou teste, economizando memória VRAM e tempo de GPU.

---

### 🧭 Trilha da Aula
1. [Carregamento do MNIST e Achatamento em NumPy](#1-carregamento-do-dataset-mnist)
2. [Redimensionamento Flat (784 Atributos)](#2-achatamento-para-784-pixels)
3. [Gerenciamento de Dispositivo (CPU vs GPU CUDA)](#3-configuração-do-dispositivo-de-hardware)
4. [Tensores, TensorDataset e DataLoader](#4-tensores-pytorch-e-dataloaders)
5. [Arquitetura Neural com nn.Module (BasicTorchNN)](#5-construção-do-modelo-com-nnmodule)
6. [Loop de Treinamento Explícito & Avaliação no Teste](#6-laço-de-treinamento-e-teste-em-pytorch)
7. [Encapsulador Scikit-Learn: TorchWrappedNN](#7-wrapper-compatível-com-scikit-learn)
8. [Pipeline Integrado com Divide255 e Flatten](#8-pipeline-completo-com-pytorch-e-scikit-learn)


## 1. Carregamento do Dataset MNIST

**O que estamos fazendo**: Carregar o conjunto MNIST oficial contendo 60.000 imagens de treino e 10.000 de teste.

**Por que fazer assim**: Utilizamos o carregador do Keras/TensorFlow para obter os arrays NumPy brutos de forma padronizada.



In [ ]:
# Importa a função utilitária para download e extração do MNIST
from tensorflow.keras.datasets import mnist

# Carrega as imagens e rótulos originais em matrizes NumPy
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print('Dimensão original de treino:', train_images.shape)
print('Dimensão original de teste: ', test_images.shape)


## 2. Achatamento para 784 Pixels

**O que estamos fazendo**: Converter cada imagem $28 \times 28$ em um vetor unidimensional de $784$ elementos contínuos.

**Por que fazer assim**: Para a rede MLP Flat, as entradas de cada amostra devem ser vetores 1D de características.

> [!NOTE]
> O formato passa de `(60000, 28, 28)` para `(60000, 784)`.



In [ ]:
# Achata os tensores 2D para vetores 1D de 784 posições
train_flat_images = train_images.reshape((-1, 28 * 28))
test_flat_images = test_images.reshape((-1, 28 * 28))

print('Formato achatado de treino:', train_flat_images.shape)
print('Formato achatado de teste: ', test_flat_images.shape)


## 3. Configuração do Dispositivo de Hardware (Device)

**O que estamos fazendo**: Verificar se existe placa de vídeo (GPU) com aceleração CUDA disponível; caso contrário, utilizar o processador (CPU).

**Por que fazer assim**: Garante portabilidade: o mesmo código roda com máxima aceleração em máquinas com GPU ou em computadores convencionais com CPU.

> [!TIP]
> `torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')` é a convenção padrão em projetos profissionais de PyTorch.



In [ ]:
import torch

# Detecta automaticamente se há GPU Nvidia com suporte a CUDA disponível
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de execução selecionado: {device}')


## 4. Tensores PyTorch e DataLoaders

**O que estamos fazendo**: Converter os arrays NumPy para tensores PyTorch e criar os objetos `TensorDataset` e `DataLoader`.

**Por que fazer assim**: O PyTorch exige tensores com tipos de dados explícitos: entradas devem ser `torch.float32` e rótulos inteiros de classes devem ser `torch.long` (int64).

> [!NOTE]
> `shuffle=True` no `train_loader` garante que as amostras de treino sejam reembaralhadas a cada nova época, prevenindo viés de ordem dos dados!



In [ ]:
# Módulos de rede neural, otimização e utilitários de dados do PyTorch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Converte matrizes NumPy para tensores e transfere para a memória do dispositivo (CPU ou GPU)
train_flat_images_tensor = torch.from_numpy(train_flat_images).float().to(device)
train_labels_tensor = torch.from_numpy(train_labels).long().to(device)
test_flat_images_tensor = torch.from_numpy(test_flat_images).float().to(device)
test_labels_tensor = torch.from_numpy(test_labels).long().to(device)

# Cria os datasets combinando tensores de características (X) e rótulos (y)
train_dataset = TensorDataset(train_flat_images_tensor, train_labels_tensor)
test_dataset = TensorDataset(test_flat_images_tensor, test_labels_tensor)

# DataLoaders para iteração em minibatches de 128 amostras
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)


## 5. Construção do Modelo com nn.Module

**O que estamos fazendo**: Definir a classe `BasicTorchNN(nn.Module)` com duas camadas lineares (`nn.Linear`) e ativação `F.relu`, e executar o laço de treinamento explícito por 5 épocas.

**Por que fazer assim**: Apresenta a estrutura orientada a objetos canônica do PyTorch, onde instanciamos camadas no construtor `__init__` e definimos o grafo de execução em `forward(self, x)`.

> [!NOTE]
> A camada final retorna *logits* brutos (sem Softmax) porque o `nn.CrossEntropyLoss()` do PyTorch já integra internamente o cálculo estável da Softmax e Log-Likelihood!

> [!IMPORTANT]
> No loop de treino, os 4 passos obrigatórios são: 1. `optimizer.zero_grad()`, 2. `outputs = model(...)`, 3. `loss.backward()`, 4. `optimizer.step()`.



In [ ]:
class BasicTorchNN(nn.Module):
  """
  Rede Neural Multicamadas (MLP Flat) em PyTorch.
  Entrada: 784 pixels -> Oculta: 512 neurônios (ReLU) -> Saída: num_classes logits.
  """
  def __init__(self, num_classes):
    super(BasicTorchNN, self).__init__()
    # Camada densa 1: mapeia 784 atributos para 512 ativações latentes
    self.fc1 = nn.Linear(28 * 28, 512)
    # Camada densa 2: mapeia 512 ativações para os logits das classes de saída
    self.fc2 = nn.Linear(512, num_classes)

  def forward(self, x):
    # Propagação direta: Linear -> ReLU -> Linear
    x = self.fc1(x)
    x = F.relu(x)
    x = self.fc2(x)
    return x

# Instancia a rede e aloca no dispositivo
model = BasicTorchNN(num_classes=10).to(device)
# Otimizador Adam com taxa de aprendizado de 0.001
optimizer = optim.Adam(model.parameters(), lr=0.001)
# Função de perda de entropia cruzada para classificação multiclasse
criterion = nn.CrossEntropyLoss()

# Laço explícito de épocas de treinamento
epochs = 5
for epoch in range(epochs):
  for batch_idx, (images, labels) in enumerate(train_loader):
    # 1. Zera os gradientes acumulados na iteração anterior
    optimizer.zero_grad()
    # 2. Forward pass: calcula as saídas da rede
    outputs = model(images)
    # 3. Calcula o custo
    loss = criterion(outputs, labels)
    # 4. Backward pass: calcula os gradientes analíticos via autodiferenciação
    loss.backward()
    # 5. Atualiza os pesos aplicando o algoritmo Adam
    optimizer.step()
  print(f'Época [{epoch+1}/{epochs}] - Perda Final do Lote: {loss.item():.4f}')

# Avaliação no conjunto de teste sob contexto sem gradientes (torch.no_grad)
with torch.no_grad():
  correct = 0
  total = 0
  for data in test_loader:
    images, labels = data
    outputs = model(images)
    # Obtém o índice da maior probabilidade/logit
    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
  print(f'Acurácia no Teste (PyTorch): {correct / total * 100:.2f}%')


## 6. Wrapper Compatível com Scikit-Learn: TorchWrappedNN

**O que estamos fazendo**: Criar uma classe que encapsula a rede PyTorch e herda de `BaseEstimator` e `ClassifierMixin`.

**Por que fazer assim**: Permite utilizar o modelo PyTorch como qualquer classificador comum do Scikit-Learn com métodos `.fit(X, y)` e `.predict(X)`.

> [!TIP]
> No método `predict`, usamos `predicted.cpu().numpy()` para trazer os tensores da GPU de volta para a CPU como arrays NumPy padrão!



In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
import numpy as np

class TorchWrappedNN(BaseEstimator, ClassifierMixin):
  """
  Wrapper para redes neurais PyTorch compatível com a API do Scikit-Learn.
  """
  def __init__(self, epochs, batch_size, model_fabric=BasicTorchNN):
    self.epochs = epochs
    self.batch_size = batch_size
    self.model_fabric = model_fabric

  def fit(self, X, y):
    # Identifica classes únicas e índices
    self.labels, ids = np.unique(y, return_inverse=True)
    # Converte arrays de entrada para tensores no dispositivo
    X_tensor = torch.from_numpy(X).float().to(device)
    y_tensor = torch.from_numpy(y).long().to(device)
    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

    # Instancia a arquitetura e otimizador
    self.model = self.model_fabric(len(self.labels)).to(device)
    self.optimizer = optim.Adam(self.model.parameters(), lr=0.001)
    self.criterion = nn.CrossEntropyLoss()

    # Executa o loop de treinamento
    for epoch in range(self.epochs):
      for batch_idx, (images, labels) in enumerate(loader):
        self.optimizer.zero_grad()
        outputs = self.model(images)
        loss = self.criterion(outputs, labels)
        loss.backward()
        self.optimizer.step()
      print(f'Época [{epoch+1}/{self.epochs}] - Perda: {loss.item():.4f}')
    return self

  def predict(self, X):
    # Inferência no conjunto de teste sem construir grafos de gradientes
    with torch.no_grad():
      X_tensor = torch.tensor(X, dtype=torch.float32).to(device)
      outputs = self.model(X_tensor)
      _, predicted = torch.max(outputs.data, 1)
      # Mapeia os índices previstos de volta para as classes originais
      return self.labels[predicted.cpu().numpy()]

# Testa o estimador com X_train e X_test achatados
model = TorchWrappedNN(epochs=5, batch_size=128)
model.fit(train_flat_images, train_labels)
predictions = model.predict(test_flat_images)
print(f'Acurácia TorchWrappedNN: {accuracy_score(test_labels, predictions) * 100:.2f}%')


## 7. Pipeline Completo: Divide255 -> Flatten -> PyTorch

**O que estamos fazendo**: Construir um `Pipeline` integrando a divisão por 255.0, o transformador `Flatten` e o modelo PyTorch.

**Por que fazer assim**: Permite passar as imagens 2D brutas diretamente para o pipeline sem nenhum passo manual externo de formatação.

> [!TIP]
> Esse padrão modular garante que o código seja limpo, elegante e à prova de falhas em produção!



In [ ]:
from sklearn.base import TransformerMixin
from sklearn.pipeline import Pipeline

class Divide255(TransformerMixin):
  """
  Normaliza tensores dividindo por 255.0.
  """
  def fit(self, X, y=None):
    return self

  def transform(self, X, y=None):
    return X / 255.0

class Flatten(TransformerMixin):
  """
  Achata matrizes bidimensionais para vetores 1D por amostra.
  """
  def fit(self, X, y=None):
    return self

  def transform(self, X, y=None):
    return X.reshape((X.shape[0], -1))

# Monta o pipeline completo de ponta a ponta com PyTorch
model = Pipeline([
  ('scaler', Divide255()),
  ('flatten', Flatten()),
  ('nn', TorchWrappedNN(epochs=5, batch_size=128))
])

# Treinamento direto com imagens 2D originais
model.fit(train_images, train_labels)
y_pred = model.predict(test_images)
print(f'Acurácia do Pipeline Completo (PyTorch): {accuracy_score(test_labels, y_pred) * 100:.2f}%')


---

## 🎓 Resumo da Aula & O Que Você Aprendeu

Nesta aula, realizamos a implementação do classificador MLP Flat no ecossistema **PyTorch**, conectando-o ao Scikit-Learn:

1. **Fundamentos do PyTorch**: Aprendemos a definir modelos como classes derivadas de `nn.Module`, implementando `__init__` e a propagação direta `forward(self, x)`.
2. **Pipelines de Dados com DataLoaders**: Vimos como criar `TensorDataset` e empacotar lotes eficientes com `DataLoader` para embaralhar o treino a cada época.
3. **Loss Sem Softmax Explícita**: Entendemos por que o `nn.CrossEntropyLoss` opera sobre *logits* brutos garantindo estabilidade numérica contra overflow.
4. **Laço de Treinamento Explícito**: Dominamos os 4 passos canônicos do gradiente descendente no PyTorch (`zero_grad`, forward, `loss.backward`, `optimizer.step`).
5. **Wrapper e Pipelines no Scikit-Learn**: Construímos a classe `TorchWrappedNN` e conectamos imagens 2D diretamente via `Pipeline([('scaler', Divide255()), ('flatten', Flatten()), ('nn', ...)])`.

---

## 👉 Próximo Passo
Na **Aula 07d**, completaremos a trilha de visão computacional implementando **Redes Neurais Convolucionais (CNNs) em PyTorch** com `nn.Conv2d` e `nn.MaxPool2d`!
